In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [7]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [8]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Kevin Durant,Over,26.5,-137,2025-11-20,2025-11-19T23:42:57Z
1,Underdog,player_points,Kevin Durant,Under,26.5,-137,2025-11-20,2025-11-19T23:42:57Z
2,Underdog,player_points,Donovan Mitchell,Over,27.5,-137,2025-11-20,2025-11-19T23:42:57Z
3,Underdog,player_points,Donovan Mitchell,Under,27.5,-137,2025-11-20,2025-11-19T23:42:57Z
4,Underdog,player_points,Alperen Sengun,Over,22.5,-137,2025-11-20,2025-11-19T23:42:57Z


### Update projected starting lineups

In [5]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\PRODUCTION/teamInfo.py
Updated 18 teams with confirmed lineups


### Top EVs for single bets

In [9]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 109 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
0,Josh Giddey,BetRivers,21.5,25.27,Over,120,0,5.06,0.421,High
1,Dereck Lively II,BetMGM,4.5,7.96,Over,-125,0,4.68,0.585,Low
2,Josh Giddey,BetRivers,20.5,25.27,Over,102,1,4.51,0.442,High
3,Pelle Larsson,BetRivers,10.5,13.30,Over,112,0,4.37,0.390,High
4,Landry Shamet,BetRivers,10.5,13.84,Over,100,0,4.20,0.420,High


## Top EVs for 2 leg bets

### Underdog picks

In [10]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 90 players...
Error getting prediction for Coby White: float division by zero
Processing 83 players with valid predictions...
Generated 3227 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 124 combinations from 3227 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Josh Giddey,Jerami Grant,19.5,23.5,25.27,17.27,over,under,1,6.43,0.321,High,High
1,Jerami Grant,Isaac Okoro,23.5,5.5,17.27,9.04,under,over,0,6.33,0.316,High,High
2,Reed Sheppard,Jerami Grant,10.5,23.5,14.27,17.27,over,under,0,5.88,0.294,Med,High
3,Reed Sheppard,Isaac Okoro,10.5,5.5,14.27,9.04,over,over,0,5.82,0.291,Med,High
4,Reed Sheppard,Josh Giddey,10.5,19.5,14.27,25.27,over,over,0,5.75,0.287,Med,High
5,Landry Shamet,Isaac Okoro,9.5,5.5,13.84,9.04,over,over,0,5.70,0.285,High,High
6,Landry Shamet,Josh Giddey,9.5,19.5,13.84,25.27,over,over,0,5.56,0.278,High,High
7,Jeremiah Fears,Landry Shamet,13.5,9.5,18.11,13.84,over,over,0,5.08,0.254,High,High
8,Jeremiah Fears,Kevin Huerter,13.5,10.5,18.11,14.35,over,over,0,4.52,0.226,High,High
9,Aaron Gordon,Jeremiah Fears,17.5,13.5,22.11,18.11,over,over,1,4.45,0.223,High,High


### Prizepicks picks

In [11]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 130 players...


c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\PRODUCTION\pipeline.py:309: RuntimeWarning: invalid value encountered in scalar divide
  res.append(player_df['PTS'].tail(5).mean() / player_df['PTS'].tail(20).mean())
c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\PRODUCTION\pipeline.py:319: RuntimeWarning: invalid value encountered in scalar divide
  res.append(calculate_volatility(player_df, 'PTS', 10) / calculate_volatility(player_df, 'PTS', 40))
c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\PRODUCTION\pipeline.py:322: RuntimeWarning: invalid value encountered in scalar divide
  res.append(calculate_volatility(player_df, 'TS_PCT', 10) / calculate_volatility(player_df, 'TS_PCT', 40))


Error getting prediction for Coby White: float division by zero
Processing 120 players with valid predictions...
Generated 6785 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 179 combinations from 6785 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Zion Williamson,Isaac Okoro,18.5,5.5,22.59,9.04,over,over,0,5.93,0.296,Med,High
1,Landry Shamet,Josh Giddey,9.5,19.5,13.84,25.27,over,over,0,5.85,0.293,High,High
2,Landry Shamet,Isaac Okoro,9.5,5.5,13.84,9.04,over,over,0,5.75,0.288,High,High
3,Tony Bradley,Isaac Okoro,4.5,5.5,7.06,9.04,over,over,0,5.73,0.287,Low,High
4,Zion Williamson,Landry Shamet,18.5,9.5,22.59,13.84,over,over,0,5.73,0.287,Med,High


## 3 leg parlay

### Underdog picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 90 players...
Error getting prediction for Coby White: float division by zero
Processing 83 players with valid predictions...
Generated 90858 valid 3-leg combinations


### Prizepicks picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 131 players...
Error getting prediction for Coby White: float division by zero
Processing 121 players with valid predictions...
Generated 284848 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 80 combinations from 284848 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Dereck Lively II,Josh Giddey,Isaac Okoro,4.5,19.5,5.5,7.96,25.27,9.04,over,over,over,0,13.30,0.266,Low,High,High
1,Zion Williamson,Dereck Lively II,Isaac Okoro,18.5,4.5,5.5,22.59,7.96,9.04,over,over,over,0,13.09,0.262,Med,Low,High
2,Tony Bradley,Zion Williamson,Josh Giddey,4.5,18.5,19.5,7.06,22.59,25.27,over,over,over,0,11.30,0.226,Low,Med,High
3,Tony Bradley,Landry Shamet,Mitchell Robinson,4.5,9.5,4.5,7.06,13.84,6.89,over,over,over,0,10.70,0.214,Low,High,Med
4,Landry Shamet,Mitchell Robinson,Jerami Grant,9.5,4.5,22.5,13.84,6.89,17.27,over,over,under,0,10.41,0.208,High,Med,High
